# Transform univariate and multivariate data to be able to be used as an input for CNNs or LSTMs

Reference: Jason Brownlee. "Deep Learning for Time Series Forecasting: Predict the Future with MLPs, CNNs, and LSTMs in Python".

In [8]:
# transform univariate time series to supervised learning problem
from numpy import array

# split a univariate sequence into samples
def split_sequence(sequence, n_steps):
    X, y = list(), list()
    for i in range(len(sequence)):
        # find the end of this pattern
        end_ix = i + n_steps
        # check if we are beyond the sequence
        if end_ix > len(sequence)-1:
            break
        # gather input and output parts of the pattern
        seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
        X.append(seq_x)
        y.append(seq_y)
    return array(X), array(y)

# define univariate time series
series = array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print(series.shape)
# transform to a supervised learning problem
X, y = split_sequence(series, 3)
print(X.shape, y.shape)
# show each sample
for i in range(len(X)):
    print(X[i], y[i])

(10,)
(7, 3) (7,)
[1 2 3] 4
[2 3 4] 5
[3 4 5] 6
[4 5 6] 7
[5 6 7] 8
[6 7 8] 9
[7 8 9] 10



> Preparing time series data for CNNs and LSTMs requires one additional step beyond transforming the data into a supervised learning problem. This one additional step causes the most confusion for beginners. In this section we will slowly step through the basics of how and why we need to prepare three-dimensional data for CNNs and LSTMs before working through an example in the next section.

> The input layer for CNN and LSTM models is specified by the input shape argument on the first hidden layer of the network. This too can make things confusing for beginners as intuitively we may expect the first layer defined in the model be the input layer, not the first hidden layer. For example, below is an example of a network with one hidden LSTM layer and one Dense output layer.

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

# 1. Define the model
model = Sequential()

# 2. Add an explicit Input layer (modern Keras standard)
# We pass our (Time Steps, Features) shape here
model.add(Input(shape=(3, 1)))

# 3. Add your LSTM and Dense layers
model.add(LSTM(32))
model.add(Dense(1))

# 4. Compile the model
model.compile(optimizer='adam', loss='mse')

Model successfully built and compiled (with no warnings)!


>In this example, the `LSTM()` layer must specify the shape of the input data. The input to every CNN and LSTM layer must be three-dimensional. The three dimensions of this input are:
>* Samples. One sequence is one sample. A batch is comprised of one or more samples.
>* Time Steps. One time step is one point of observation in the sample. One sample is comprised of multiple time steps.
>* Features. One feature is one observation at a time step. One time step is comprised of one or more features.

>This expected three-dimensional structure of input data is often summarized using the array shape notation of: `[samples, timesteps, features]`. Remember, that the two-dimensional shape of a dataset that we are familiar with from the previous section has the array shape of: `[samples, features]`. This means we are adding the new dimension of **time steps**. Except, in time series forecasting problems our features are observations at time steps. So, really, we are adding the **dimension of features**, where a univariate time series has only one feature.

In [9]:
# transform univariate time series to supervised learning problem
from numpy import array

# split a univariate sequence into samples
def split_sequence(sequence, n_steps):
    X, y = list(), list()
    for i in range(len(sequence)):
        # find the end of this pattern
        end_ix = i + n_steps
        # check if we are beyond the sequence
        if end_ix > len(sequence)-1:
            break
        # gather input and output parts of the pattern
        seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
        X.append(seq_x)
        y.append(seq_y)
    return array(X), array(y)

# define univariate time series
series = array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print(series.shape)
# transform to a supervised learning problem
X, y = split_sequence(series, 3)
print(X.shape, y.shape)
# transform input from [samples, features] to [samples, timesteps, features]
X = X.reshape((X.shape[0], X.shape[1], 1))
print(X.shape)

(10,)
(7, 3) (7,)
(7, 3, 1)


> Running the example first prints the shape of the univariate time series, in this case 10
time steps. It then summarizes the shape if the input ($X$) and output ($y$) elements of each
sample after the univariate series has been converted into a supervised learning problem, in
this case, the data has 7 samples and the input data has 3 features per sample, which we
know are actually time steps. Finally, the input element of each sample is reshaped to be
three-dimensional suitable for fitting an LSTM or CNN and now has the shape `[7, 3, 1]` or 7
samples, 3 time steps, 1 feature.

>*I have two columns in my data file with 5,000 rows, column 1 is time (with 1 hour interval) and column 2 is the number of sales and I am trying to forecast the number of sales for future time steps. Help me to set the number of samples, time steps and features in this data for an LSTM?* 

>There are few problems here:

>* Data Shape. LSTMs expect 3D input, and it can be challenging to get your head around this the first time.
>* Sequence Length. LSTMs don’t like sequences of more than 200-400 time steps, so the data will need to be split into subsamples.

>We will work through this example, broken down into the following 4 steps:
>1. Load the Data
>2. Drop the Time Column
>3. Split Into Samples
>4. Reshape Subsequences

In [11]:
# example of defining a dataset
from numpy import array
# define the dataset
data = list()
n = 5000
for i in range(n):
    data.append([i+1, (i+1)*10])
data = array(data)
print(data[:5, :])
print(data.shape)

[[ 1 10]
 [ 2 20]
 [ 3 30]
 [ 4 40]
 [ 5 50]]
(5000, 2)


>If your time series data is uniform over time and there is no missing values, we can drop the time column. If not, you may want to look at imputing the missing values, resampling the data to a new time scale, or developing a model that can handle missing values. Here, we just drop the first column:

In [12]:
# example of dropping the time dimension from the dataset
from numpy import array
# define the dataset
data = list()
n = 5000
for i in range(n):
    data.append([i+1, (i+1)*10])
data = array(data)
# drop time
data = data[:, 1]
print(data.shape)

(5000,)


>LSTMs need to process samples where each sample is a single sequence of observations. In this
case, 5,000 time steps is too long; LSTMs work better with 200-to-400 time steps. Therefore, we
need to split the 5,000 time steps into multiple shorter sub-sequences. There are many ways to
do this, and you may want to explore some depending on your problem. For example, perhaps
you need overlapping sequences, perhaps non-overlapping is good but your model needs state
across the sub-sequences and so on. In this example, we will split the 5,000 time steps into 25
sub-sequences of 200 time steps each. Rather than using NumPy or Python tricks, we will do
this the old fashioned way so you can see what is going on.

In [13]:
# example of splitting a univariate sequence into subsequences
from numpy import array
# define the dataset
data = list()
n = 5000
for i in range(n):
    data.append([i+1, (i+1)*10])
data = array(data)
# drop time
data = data[:, 1]
# split into samples (e.g. 5000/200 = 25)
samples = list()
length = 200
# step over the 5,000 in jumps of 200
for i in range(0,n,length):
    # grab from i to i + 200
    sample = data[i:i+length]
    samples.append(sample)
print(len(samples))

25


>The LSTM needs data with the format of [samples, timesteps, features]. We have 25
samples, 200 time steps per sample, and 1 feature. First, we need to convert our list of arrays
into a 2D NumPy array with the shape [25, 200].

In [15]:
# example of creating an array of subsequence
from numpy import array
# define the dataset
data = list()
n = 5000
for i in range(n):
    data.append([i+1, (i+1)*10])
data = array(data)
# drop time
data = data[:, 1]
# split into samples (e.g. 5000/200 = 25)
samples = list()
length = 200
# step over the 5,000 in jumps of 200
for i in range(0,n,length):
    # grab from i to i + 200
    sample = data[i:i+length]
    samples.append(sample)
# convert list of arrays into 2d array
data = array(samples)
print(data.shape)

(25, 200)


>Running this piece, you should see that we have 25 rows and 200 columns. Interpreted in a
machine learning context, this dataset has 25 samples and 200 features per sample. Next, we can use the `reshape()` function to add one additional dimension for our single
feature and use the existing columns as time steps instead.

In [16]:
# example of creating a 3d array of subsequences
from numpy import array
# define the dataset
data = list()
n = 5000
for i in range(n):
    data.append([i+1, (i+1)*10])
data = array(data)
# drop time
data = data[:, 1]
# split into samples (e.g. 5000/200 = 25)
samples = list()
length = 200
# step over the 5,000 in jumps of 200
for i in range(0,n,length):
    # grab from i to i + 200
    sample = data[i:i+length]
    samples.append(sample)
# convert list of arrays into 2d array
data = array(samples)
# reshape into [samples, timesteps, features]
data = data.reshape((len(samples), length, 1))
print(data.shape)

(25, 200, 1)


And that is it. The data can now be used as an input ($X$) to an LSTM model, or even a CNN model.

*So what is a "feature"?* Let's define it in the context of the examples above. 
* **Example 1 (The 1 to 10 sequence):** You are only tracking one sequence of numbers. Therefore, you have **1 feature** (the value of the number itself). 
* **Example 2 (The Sales Data):** Your data consists of "Time" and "Sales". Because "Time" is just the index (the timeline keeping everything in order), the only actual variable you are feeding the neural network to learn from is "Sales". Therefore, you have **1 feature**. 

Because both examples only track a single variable over time, they are called **Univariate** time series.

Let's introduce multiple features: Instead of just recording the number of sales every hour, you also recorded the **Temperature** outside and the **Ad Spend** for that hour. 

Now, at every single time step, you are recording three distinct pieces of information:
1.  Sales
2.  Temperature
3.  Ad Spend

This is called a **Multivariate** time series. If you prepared this new dataset using the exact same sliding window technique from your notebook, your final data shape would change from `[25, 200, 1]` to `[25, 200, 3]`. The `3` represents your three features.

So, when an LSTM asks for an input shape of `(200, 1)`, it is simply asking: *"How many rows (time steps) am I looking at at once, and how many columns (features) of data do I need to read across those rows?"*

In [18]:
import numpy as np
import random

# Define the dataset with multiple variables
data = list()
n = 5000

for i in range(n):
    time_step = i + 1
    sales = (i + 1) * 10
    temperature = 20 + random.randint(-5, 5)  # Simulated temperature data
    ad_spend = (i + 1) * 5                    # Simulated ad spend data
    
    # Append a single row with 4 columns: [Time, Sales, Temperature, Ad Spend]
    data.append([time_step, sales, temperature, ad_spend])

data = np.array(data)
print(f"Original dataset shape: {data.shape} -> (Time Steps, Total Columns)")

# Drop the time column (index 0) 
# We use data[:, 1:] to say "give us all rows, and all columns starting from index 1 to the end"
data = data[:, 1:]
print(f"Shape after dropping time: {data.shape} -> (Time Steps, Features)")

# Split into samples (e.g. 5000 / 200 = 25 samples)
samples = list()
length = 200

# Step over the 5,000 rows in jumps of 200
for i in range(0, n, length):
    # Grab from i to i + 200. 
    # Because 'data' now has 3 columns, each 'sample' is a 2D grid of shape (200, 3)
    sample = data[i:i+length, :]
    samples.append(sample)

# Convert the list of 2D grids into a 3D array
data = np.array(samples)

print(f"\nFinal 3D Array Shape: {data.shape}")
print("Meaning: [Samples, Time Steps, Features]")

Original dataset shape: (5000, 4) -> (Time Steps, Total Columns)
Shape after dropping time: (5000, 3) -> (Time Steps, Features)

Final 3D Array Shape: (25, 200, 3)
Meaning: [Samples, Time Steps, Features]
